# Movie Search App (sakila)

A console app that searches films in the **sakila** database (MySQL) and logs
every search to **MongoDB**. Each section below can be expanded by clicking its heading.

## Imports

In [1]:
%pip install pymysql pymongo python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
import pymysql
from pymongo import MongoClient
from pymongo.errors import PyMongoError
from collections import Counter
from datetime import datetime

load_dotenv()  # loads MYSQL_* and MONGO_URI from .env


## 1. Connecting to MySQL and the sakila database

In [4]:
# MySQL connection settings.
# Credentials are read from the .env file (it is not committed to git, see .env.example).
config = {
    "host": os.getenv("MYSQL_HOST"),
    "user": os.getenv("MYSQL_USER"),
    "password": os.getenv("MYSQL_PASSWORD"),
}
db_name = "sakila"


def run_query(sql, params=None):
    """Opens a connection, selects the sakila database, runs the query and
    returns all rows. Connecting and selecting the database live here so they
    are not repeated in every search function.
    Parameters are passed via %s, which protects against SQL injection."""
    with pymysql.connect(**config) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"USE {db_name}")
            cursor.execute(sql, params or ())
            return cursor.fetchall()

In [5]:
# Connection check.
try:
    with pymysql.connect(**config) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"USE {db_name}")
    print("MySQL connection successful!")
except pymysql.MySQLError as error:
    print("Could not connect to MySQL.")
    print("Check the connection settings (host, user, password) and network access.")
    print(f"Details: {error}")

MySQL connection successful!


## 2. Search functions (MySQL)

In [6]:
def fetch_films_by_keyword(keyword):
    """Films whose title contains keyword.
    Returns (title, genre, year, rating) via a JOIN with category,
    so the output matches the other search modes."""
    return run_query(
        """
        SELECT f.title, c.name, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        WHERE f.title LIKE %s
        ORDER BY f.title
        """,
        ("%" + keyword + "%",),
    )

In [7]:
def fetch_genres():
    rows = run_query("SELECT name FROM category ORDER BY name")
    return [row[0] for row in rows]
def fetch_year_range():
    """Earliest and latest release year in the database: (min_year, max_year)."""
    return run_query("SELECT MIN(release_year), MAX(release_year) FROM film")[0]

In [8]:
def fetch_films_by_genre_year(genre, start_year, end_year):
    """Films of the given genre released in [start_year; end_year].
    Returns (title, genre, year, rating)."""
    return run_query(
        """
        SELECT f.title, c.name, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        WHERE c.name = %s AND f.release_year BETWEEN %s AND %s
        ORDER BY f.release_year, f.title
        """,
        (genre, start_year, end_year),
    )

In [9]:
def fetch_ratings():
    """Age ratings available in the database (G, PG, PG-13, R, NC-17)."""
    rows = run_query(
        "SELECT DISTINCT rating FROM film WHERE rating IS NOT NULL ORDER BY rating"
    )
    return [row[0] for row in rows]


def fetch_films_by_rating(rating):
    """Films with the given age rating (bonus task).
    Returns (title, genre, year, rating)."""
    return run_query(
        """
        SELECT f.title, c.name, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        WHERE f.rating = %s
        ORDER BY f.title
        """,
        (rating,),
    )

## 3. Paginated output

In [10]:
PAGE_SIZE = 10  # how many results to show at a time

HEADERS = ("No", "Title", "Genre", "Year", "Rating")  # Output table column headers.

def paginate(rows):
    """Prints rows as a table, PAGE_SIZE rows at a time.
    Each row is (title, genre, year, rating).
    Table columns: No, Title, Genre, Year, Rating."""
    if not rows:
        print("Nothing found for your search.")
        return

    total = len(rows)

    # Each column is as wide as its longest value (or header), so the
    # table lines up. Done by hand, without third-party modules.
    num_width = max(len(HEADERS[0]), len(str(total)))
    title_width = max(len(HEADERS[1]), max(len(str(row[0])) for row in rows))
    genre_width = max(len(HEADERS[2]), max(len(str(row[1])) for row in rows))
    year_width = max(len(HEADERS[3]), max(len(str(row[2])) for row in rows))
    rating_width = max(len(HEADERS[4]), max(len(str(row[3])) for row in rows))

    # Table header and separator line.
    print(
        f"{HEADERS[0]:>{num_width}}  {HEADERS[1]:<{title_width}}  "
        f"{HEADERS[2]:<{genre_width}}  {HEADERS[3]:>{year_width}}  "
        f"{HEADERS[4]:<{rating_width}}"
    )
    print(
        f"{'-' * num_width}  {'-' * title_width}  {'-' * genre_width}  "
        f"{'-' * year_width}  {'-' * rating_width}"
    )

    shown = 0
    while shown < total:
        # Row numbers continue across pages.
        for number, row in enumerate(rows[shown:shown + PAGE_SIZE], start=shown + 1):
            title, genre, year, rating = row
            print(
                f"{number:>{num_width}}  {str(title):<{title_width}}  "
                f"{str(genre):<{genre_width}}  {str(year):>{year_width}}  "
                f"{str(rating):<{rating_width}}"
            )
        shown += min(PAGE_SIZE, total - shown)
        if shown < total:
            answer = input(
                f"\nShown {shown} of {total}. "
                f"Show more? (Enter = yes, n = no): "
            )
            if answer.strip().lower() == "n":
                break

## 4. Connecting to my own MongoDB database

In [11]:
# The MongoDB address with user and password is stored in .env (MONGO_URI).
mongo_client = MongoClient(os.getenv("MONGO_URI"))
GROUP = "051225"
FULL_NAME = "YerkovskaYuliia"
searches_collection = mongo_client["ich_edit"][f"final_project_{GROUP}_{FULL_NAME}"]

In [12]:
try:
    mongo_client.admin.command("ping")
    print("MongoDB connection successful!")
    print("Collection:", searches_collection.name)
except PyMongoError as error:
    print("Could not connect to MongoDB.")
    print("Check the connection string (user, password, address) and network access.")
    print(f"Details: {error}")

MongoDB connection successful!
Collection: final_project_051225_YerkovskaYuliia


## 5. Logging searches to MongoDB

In [13]:
def log_search(search_type, params, results_count):
    """Saves one search to MongoDB.

    search_type   - 'keyword', 'genre_year' or 'rating'
    params        - dict with the search parameters
    results_count - how many films were found
    """
    searches_collection.insert_one({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "search_type": search_type,
        "params": params,
        "results_count": results_count,
    })

In [14]:
def _query_label(doc):
    """Turns a search document into a readable 'type: parameters' string."""
    params = doc.get("params", {})
    params_text = ", ".join(f"{key}={value}" for key, value in params.items())
    return f"{doc.get('search_type')}: {params_text}"


def show_statistics():
    """Most frequent searches and the latest unique searches."""
    docs = list(searches_collection.find())
    if not docs:
        print("No saved searches yet.")
        return

    # Most frequent searches, via Counter.
    counter = Counter(_query_label(doc) for doc in docs)
    print("\nTop 5 most frequent searches:")
    for i, (label, count) in enumerate(counter.most_common(5), start=1):
        print(f"{i}. {label}: {count} time(s)")

    # Latest unique searches: walk from the newest and skip repeats,
    # so identical searches in a row do not fill the whole list.
    recent = searches_collection.find().sort("timestamp", -1)
    print("\nLatest 5 unique searches:")
    seen = set()
    shown = 0
    for doc in recent:
        label = _query_label(doc)
        if label in seen:
            continue
        seen.add(label)
        shown += 1
        print(f"{shown}. [{doc.get('timestamp')}] {label}")
        if shown == 5:
            break

## 6. Input and validation

In [15]:
def ask_year(prompt):
    """Asks for a year (integer) and repeats on invalid input."""
    while True:
        value = input(prompt).strip()
        if value.isdigit():
            return int(value)
        print("Please enter the year as a number. Try again.")


def ask_index(prompt, count):
    """Asks for a list item number (1 to count) and repeats on invalid input.
    Returns a zero-based index into the list."""
    while True:
        value = input(prompt).strip()
        if value.isdigit() and 1 <= int(value) <= count:
            return int(value) - 1
        print(f"Please enter a number from 1 to {count}. Try again.")

## Search modes

In [16]:
def search_by_keyword():
    """Mode 1: search by keyword (part of the title)."""
    keyword = input("Enter a keyword (part of the title): ").strip()
    if not keyword:
        print("Empty search.")
        return
    rows = fetch_films_by_keyword(keyword)
    log_search("keyword", {"keyword": keyword}, len(rows))
    paginate(rows)

In [17]:
def search_by_genre_year():
    """Mode 2: search by genre and year range."""
    genres = fetch_genres()
    print("Available genres:")
    for i, genre_name in enumerate(genres, start=1):
        print(f"  {i}. {genre_name}")

    # The genre is picked by number, so the user cannot misspell it.
    index = ask_index("Enter the genre number: ", len(genres))
    genre = genres[index]

    min_year, max_year = fetch_year_range()
    print(f"\nRelease years in the database: {min_year} to {max_year}")

    start_year = ask_year("Enter the start year: ")
    end_year = ask_year("Enter the end year: ")
    if start_year > end_year:  # swap if entered in reverse order
        start_year, end_year = end_year, start_year

    rows = fetch_films_by_genre_year(genre, start_year, end_year)
    log_search(
        "genre_year",
        {"genre": genre, "start_year": start_year, "end_year": end_year},
        len(rows),
    )
    paginate(rows)

In [18]:
def search_by_rating():
    """Mode 3 (bonus): search by age rating."""
    ratings = fetch_ratings()
    print("Available ratings:")
    for rating_name in ratings:
        print(f"  - {rating_name}")

    rating = input("Enter a rating: ").strip().upper()
    if rating not in ratings:
        print("This rating is not in the list.")
        return

    rows = fetch_films_by_rating(rating)
    log_search("rating", {"rating": rating}, len(rows))
    paginate(rows)

## Main program with menu

In [19]:
def main_menu():
    """Main application loop with the menu."""
    while True:
        print("\n Movie search (sakila database) ")
        print("1. Search by keyword")
        print("2. Search by genre and year range")
        print("3. Search by rating")
        print("4. Search statistics")
        print("0. Exit")

        choice = input("Choose a menu item (1, 2, 3, 4 or 0): ").strip()

        try:
            if choice == "1":
                search_by_keyword()
            elif choice == "2":
                search_by_genre_year()
            elif choice == "3":
                search_by_rating()
            elif choice == "4":
                show_statistics()
            elif choice == "0":
                print("Exiting the program.")
                break
            else:
                print("Invalid choice. Choose 1, 2, 3, 4 or 0.")
        except Exception as error:
            # An error in one menu item must not crash the program.
            print(f"An error occurred: {error}")

In [20]:
### main_menu()


 Movie search (sakila database) 
1. Search by keyword
2. Search by genre and year range
3. Search by rating
4. Search statistics
0. Exit


Choose a menu item (1, 2, 3, 4 or 0):  1
Enter a keyword (part of the title):  Fam


No  Title        Genre        Year  Rating
--  --------------  ---------  ----  -------
 1  CYCLONE FAMILY  Foreign    2008  PG     
 2  DOGMA FAMILY    Animation  1995  G      
 3  FAMILY SWEET    Horror     1993  R      

 Movie search (sakila database) 
1. Search by keyword
2. Search by genre and year range
3. Search by rating
4. Search statistics
0. Exit


Choose a menu item (1, 2, 3, 4 or 0):  1
Enter a keyword (part of the title):  ghdhgd


Nothing found for your search.

 Movie search (sakila database) 
1. Search by keyword
2. Search by genre and year range
3. Search by rating
4. Search statistics
0. Exit


...


Top 5 most frequent searches:
1. rating: rating=G: 4 time(s)
2. keyword: keyword=family: 2 time(s)
3. keyword: keyword=LOVE: 2 time(s)
4. keyword: keyword=travel: 2 time(s)
5. genre_year: genre=Travel, start_year=1990, end_year=2000: 2 time(s)

Latest 5 unique searches:
1. [2026-06-18T20:18:18] rating: rating=G
2. [2026-06-18T20:17:04] genre_year: genre=Action, start_year=1990, end_year=2000
3. [2026-06-18T20:16:32] genre_year: genre=Action, start_year=19900, end_year=445646
4. [2026-06-18T20:15:44] keyword: keyword=ghdhgd
5. [2026-06-18T20:15:17] keyword: keyword=Fam

 Movie search (sakila database) 
1. Search by keyword
2. Search by genre and year range
3. Search by rating
4. Search statistics
0. Exit


Choose a menu item (1, 2, 3, 4 or 0):  4



Top 5 most frequent searches:
1. rating: rating=G: 4 time(s)
2. keyword: keyword=family: 2 time(s)
3. keyword: keyword=LOVE: 2 time(s)
4. keyword: keyword=travel: 2 time(s)
5. genre_year: genre=Travel, start_year=1990, end_year=2000: 2 time(s)

Latest 5 unique searches:
1. [2026-06-18T20:18:18] rating: rating=G
2. [2026-06-18T20:17:04] genre_year: genre=Action, start_year=1990, end_year=2000
3. [2026-06-18T20:16:32] genre_year: genre=Action, start_year=19900, end_year=445646
4. [2026-06-18T20:15:44] keyword: keyword=ghdhgd
5. [2026-06-18T20:15:17] keyword: keyword=Fam

 Movie search (sakila database) 
1. Search by keyword
2. Search by genre and year range
3. Search by rating
4. Search statistics
0. Exit


Choose a menu item (1, 2, 3, 4 or 0):  0


Exiting the program.


## Individual search cells 

###  Search by keyword

In [ ]:
search_by_keyword()

###  Search by genre and year range

In [ ]:
search_by_genre_year()

###  Search by rating (bonus task)

In [ ]:
search_by_rating()

###  Search statistics

In [ ]:
show_statistics()

In [ ]:
mongo_client.close()
print("MongoDB connection closed.")